In [7]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import requests
import re

In [8]:
def parse_college_draft(text):
    '''
    Helper function to extract draft year and college from a player card
    '''
    
    if not text:
        return None, None

    parts = [x.strip() for x in text.split(',')]

    if len(parts) == 2:
        return parts[0], parts[1]
    elif len(parts) == 1:
        return None, parts[0]
    else:
        return None, None

def replace_fractions(val: str) -> str:
    '''
    Helper fucntion to get rid of fractions in data
    '''
    for frac, dec in FRACTIONS.items():
        val = val.replace(frac, dec)
    return val


def extract_number(val: str) -> float | None:
    '''
    Strip units/symbols and return a float, or None if unparseable
    '''
    val = str(val).strip()
    if val in ("", "nan", "None", "N/A", "-"):
        return None
    val = replace_fractions(val)
    match = re.search(r"[\d]+\.?[\d]*", val)
    return float(match.group()) if match else None

def parse_height(val: str) -> float | None:
    '''
    Handle Height
    '''
    val = str(val).strip()
    if val in ("", "nan"):
        return None
    val = replace_fractions(val)
    m = re.match(r"(\d+)['\u2019](\d*\.?\d*)", val)   # e.g. 6'4 or 6'4.5
    if m:
        feet, inches = m.groups()
        return int(feet) * 12 + (float(inches) if inches else 0.0)
    m2 = re.search(r"[\d]+\.?[\d]*", val)              # bare number → inches
    return float(m2.group()) if m2 else None

In [9]:

page_url_start_year = 1999
page_url_end_year = 2026
number_of_pages = 437

In [10]:
# Loop through each page and get player_name, draft year, draf position, college, player_id

page_url_base = 'https://www.mockdraftable.com/'

player_info_dict = {}
for page_num in range(1, number_of_pages+1):
    page_url_num = f'https://www.mockdraftable.com/search?position=ATH&beginYear={page_url_start_year}&endYear={page_url_end_year}&sort=DESC&page={page_num}'
    page_text = requests.get(page_url_num).text
    soup = BeautifulSoup(page_text, 'html.parser')
    print(f'{"-"*10} PAGE {page_num} {"-"*10}')
    
    # Get table of players
    players_table = soup.find('div', class_ = 'list-group mb-2')

    # Get list of player cards from table
    players_cards = players_table.find_all(class_='list-group-item list-group-item-action justify-content-between d-flex')

    # Get Name + player link from player_cards
    for player in players_cards:
        
        split_college_draft_class = player.find('span', class_='align-middle ml-2').get_text(strip=True) # Base for college + draft
        college, draft_class = parse_college_draft(split_college_draft_class) # Get college and draft year
        player_name = player.find('h5', class_='list-group-item-heading mb-0 text-dark').get_text(strip=True) # Player's name
        clean_player_name = re.sub('[^A-Za-z0-9]+', '', player_name) # Strip out everything but letters
        player_id = clean_player_name.lower() + str(draft_class) # Create an id clean_name + draft year
        link_to_players_page = page_url_base + player.get('href') # Link to Player page
        draft_position = player.find('span', class_ ='badge').get('title') # Draft position

        # print(f'player_info | {player_name}\n\t player_id {player_id} \n\t link_to_page {link_to_players_page} \n\t draft_position {draft_position} \n\t college {college} \n\t draft_class {draft_class}')

        print(f'\tAdded {player_name}')
        player_dict = {
            'player_name': player_name,
            'player_id' : player_id,
            'player_url': link_to_players_page,
            'draft_class': int(draft_class),
            'position': draft_position,
            'college': college,
        }

        if player_id in player_info_dict:
            continue
        else:
            player_info_dict[player_id] = player_dict
    
    print(f'\tTotal Players added {len(players_cards)}\n')
    print(f'')
    

print(f'{"-"*10}\nNum of players added: {len(player_info_dict)}{"-"*10}')


    # else:


    



---------- PAGE 1 ----------
	Added A'Shawn Robinson
	Added A.C. Leonard
	Added A.J. Brown
	Added A.J. Cann
	Added A.J. Derby
	Added A.J. Edds
	Added A.J. Epenesa
	Added A.J. Green
	Added A.J. Green
	Added A.J. Greene
	Added A.J. Hawk
	Added A.J. Jefferson
	Added A.J. Jenkins
	Added A.J. Klein
	Added A.J. McCarron
	Added A.J. Nicholson
	Added A.J. Stamps
	Added A.J. Terrell
	Added A.Q. Shipley
	Added AJ Barner
	Total Players added 20


---------- PAGE 2 ----------
	Added AJ Haulcy
	Added AT Perry
	Added Aamil Wagner
	Added Aaron  Lynch
	Added Aaron Anderson
	Added Aaron Banks
	Added Aaron Brooks
	Added Aaron Burbridge
	Added Aaron Casey
	Added Aaron Colvin
	Added Aaron Corp
	Added Aaron Curry
	Added Aaron Dalan
	Added Aaron Davis
	Added Aaron Dobson
	Added Aaron Donald
	Added Aaron Fairooz
	Added Aaron Fields
	Added Aaron Francisco
	Added Aaron Fuller
	Total Players added 20


---------- PAGE 3 ----------
	Added Aaron Gibson
	Added Aaron Hansford
	Added Aaron Hester
	Added Aaron Humphr

In [11]:
display(player_info_dict)

{'ashawnrobinson2016': {'player_name': "A'Shawn Robinson",
  'player_id': 'ashawnrobinson2016',
  'player_url': 'https://www.mockdraftable.com//player/ashawn-robinson?position=ATH',
  'draft_class': 2016,
  'position': 'Defensive Tackle',
  'college': 'Alabama'},
 'acleonard2014': {'player_name': 'A.C. Leonard',
  'player_id': 'acleonard2014',
  'player_url': 'https://www.mockdraftable.com//player/ac-leonard?position=ATH',
  'draft_class': 2014,
  'position': 'Tight End',
  'college': 'Tennessee State'},
 'ajbrown2019': {'player_name': 'A.J. Brown',
  'player_id': 'ajbrown2019',
  'player_url': 'https://www.mockdraftable.com//player/aj-brown?position=ATH',
  'draft_class': 2019,
  'position': 'Wide Receiver',
  'college': 'Ole Miss'},
 'ajcann2015': {'player_name': 'A.J. Cann',
  'player_id': 'ajcann2015',
  'player_url': 'https://www.mockdraftable.com//player/aj-cann?position=ATH',
  'draft_class': 2015,
  'position': 'Offensive Guard',
  'college': 'South Carolina'},
 'ajderby2015': 

In [12]:
# Loop through each player page and extract height, wieght, arm length, hand size, 40 yard dash, vert jump, broad jump, bench press
for player_id in player_info_dict:
        player_url = player_info_dict[player_id]['player_url']
        player_name = player_info_dict[player_id]['player_name']
        player_page_text = requests.get(player_url, 'html_parser').text
        player_soup = BeautifulSoup(player_page_text)
        player_measurements_table  = player_soup.find('table', class_ = 'table table-sm mb-0')
        measurements_list = player_measurements_table.find_all('tr')

        print(f'{"-"*10} {player_name.upper()}')
        for measurement in measurements_list[1:]:
            line_items = measurement.find_all('td')
            measurement_name = line_items[0].get_text(strip=True)
            value = line_items[1].get_text(strip=True)

            print(f'{measurement_name} | {value}')
            player_info_dict[player_id][measurement_name] = value
            # print(measurement)
        # counter += 1

    # player_dict = {}


---------- A'SHAWN ROBINSON
Height | 6' 4"
Weight | 307 lbs
Arm Length | 34½"
Hand Size | 10½"
10 Yard Split | 1.78s
40 Yard Dash | 5.2s
Vertical Jump | 26"
Broad Jump | 106"
3-Cone Drill | 7.8s
20 Yard Shuttle | 4.74s
Bench Press | 22 reps
---------- A.C. LEONARD
Height | 6' 2"
Weight | 252 lbs
Arm Length | 33"
Hand Size | 9¼"
40 Yard Dash | 4.5s
Vertical Jump | 34"
Broad Jump | 128"
Bench Press | 20 reps
---------- A.J. BROWN
Height | 6' 0½"
Weight | 226 lbs
Wingspan | 78"
Arm Length | 32⅞"
Hand Size | 9¾"
40 Yard Dash | 4.49s
Vertical Jump | 36½"
Broad Jump | 120"
Bench Press | 19 reps
---------- A.J. CANN
Height | 6' 3"
Weight | 313 lbs
Arm Length | 32⅝"
Hand Size | 10¼"
Bench Press | 26 reps
---------- A.J. DERBY
Height | 6' 4"
Weight | 255 lbs
Arm Length | 30½"
Hand Size | 9½"
Bench Press | 15 reps
---------- A.J. EDDS
Height | 6' 4"
Weight | 246 lbs
Arm Length | 32¾"
Hand Size | 9⅛"
10 Yard Split | 1.6s
40 Yard Dash | 4.62s
Vertical Jump | 33"
Broad Jump | 117"
3-Cone Drill | 7.

In [13]:
display(player_info_dict)

{'ashawnrobinson2016': {'player_name': "A'Shawn Robinson",
  'player_id': 'ashawnrobinson2016',
  'player_url': 'https://www.mockdraftable.com//player/ashawn-robinson?position=ATH',
  'draft_class': 2016,
  'position': 'Defensive Tackle',
  'college': 'Alabama',
  'Height': '6\' 4"',
  'Weight': '307 lbs',
  'Arm Length': '34½"',
  'Hand Size': '10½"',
  '10 Yard Split': '1.78s',
  '40 Yard Dash': '5.2s',
  'Vertical Jump': '26"',
  'Broad Jump': '106"',
  '3-Cone Drill': '7.8s',
  '20 Yard Shuttle': '4.74s',
  'Bench Press': '22 reps'},
 'acleonard2014': {'player_name': 'A.C. Leonard',
  'player_id': 'acleonard2014',
  'player_url': 'https://www.mockdraftable.com//player/ac-leonard?position=ATH',
  'draft_class': 2014,
  'position': 'Tight End',
  'college': 'Tennessee State',
  'Height': '6\' 2"',
  'Weight': '252 lbs',
  'Arm Length': '33"',
  'Hand Size': '9¼"',
  '40 Yard Dash': '4.5s',
  'Vertical Jump': '34"',
  'Broad Jump': '128"',
  'Bench Press': '20 reps'},
 'ajbrown2019': 

In [14]:
player_name_list= []
player_id_list = []
player_url_list = []
draft_class_list = []
position_list = []
college_list = []
height_list = []
weight_list = []
arm_length_list = []
hand_size_list = []
yard_split_list = []
yard_dash_list = []
vertical_jump_list = []
broad_jump_list = []
cone_drill_list = []
yard_shuttle_list = []
bp_list= []

for player_id, player_id_dict in player_info_dict.items():
    player_name_list.append(player_id_dict.get('player_name', ''))
    player_id_list.append(player_id_dict.get('player_id', ''))
    player_url_list.append(player_id_dict.get('player_url', ''))
    draft_class_list.append(player_id_dict.get('draft_class', ''))
    position_list.append(player_id_dict.get('position', ''))
    college_list.append(player_id_dict.get('college', ''))
    height_list.append(player_id_dict.get('Height', ''))
    weight_list.append(player_id_dict.get('Weight', ''))
    arm_length_list.append(player_id_dict.get('Arm Length', ''))
    hand_size_list.append(player_id_dict.get('Hand Size', ''))
    yard_split_list.append(player_id_dict.get('10 Yard Split', ''))
    yard_dash_list.append(player_id_dict.get('40 Yard Dash', ''))
    vertical_jump_list.append(player_id_dict.get('Vertical Jump', ''))
    broad_jump_list.append(player_id_dict.get('Broad Jump', ''))
    cone_drill_list.append(player_id_dict.get('3-Cone Drill', ''))
    yard_shuttle_list.append(player_id_dict.get('20 Yard Shuttle', ''))
    bp_list.append(player_id_dict.get('Bench Press', ''))



In [15]:
nfl_combine_df = pd.DataFrame({
    'player_name' : player_name_list,
    'player_id': player_id_list,
    'player_url': player_url_list,
    'draft_class': draft_class_list,
    'position':position_list,
    'college':college_list,
    'height':height_list,
    'weight':weight_list,
    'arm_length':arm_length_list,
    'hand_size':hand_size_list,
    'yard_10_split':yard_split_list,
    'yard_40_dash': yard_dash_list,
    'vertical_jump':vertical_jump_list,
    'broad_jump':broad_jump_list,
    'cone_drill':cone_drill_list,
    'yard_20_shuttle': yard_shuttle_list,
    'bench_press': bp_list
})

In [16]:
lists = {
    "player_name_list": player_name_list,
    "player_id_list": player_id_list,
    "player_url_list": player_url_list,
    "draft_class_list": draft_class_list,
    "position_list": position_list,
    "college_list": college_list,
    "height_list": height_list,
    "weight_list": weight_list,
    "arm_length_list": arm_length_list,
    "hand_size_list": hand_size_list,
    "yard_split_list": yard_split_list,
    "yard_dash_list": yard_dash_list,
    "vertical_jump_list": vertical_jump_list,
    "broad_jump_list": broad_jump_list,
    "cone_drill_list": cone_drill_list,
    "yard_shuttle_list": yard_shuttle_list,  # fixed typo
    "bp_list": bp_list
}

for name, lst in lists.items():
    print(f"{name}: {len(lst)}")

player_name_list: 8707
player_id_list: 8707
player_url_list: 8707
draft_class_list: 8707
position_list: 8707
college_list: 8707
height_list: 8707
weight_list: 8707
arm_length_list: 8707
hand_size_list: 8707
yard_split_list: 8707
yard_dash_list: 8707
vertical_jump_list: 8707
broad_jump_list: 8707
cone_drill_list: 8707
yard_shuttle_list: 8707
bp_list: 8707


In [17]:
display(nfl_combine_df.head())

,player_name,player_id,player_url,draft_class,position,college,height,weight,arm_length,hand_size,yard_10_split,yard_40_dash,vertical_jump,broad_jump,cone_drill,yard_20_shuttle,bench_press
0,A'Shawn Robinson,ashawnrobinson2016,https://www.mockdraftable.com//player/ashawn-r...,2016,Defensive Tackle,Alabama,"6' 4""",307 lbs,"34½""","10½""",1.78s,5.2s,"26""","106""",7.8s,4.74s,22 reps
1,A.C. Leonard,acleonard2014,https://www.mockdraftable.com//player/ac-leona...,2014,Tight End,Tennessee State,"6' 2""",252 lbs,"33""","9¼""",,4.5s,"34""","128""",,,20 reps
2,A.J. Brown,ajbrown2019,https://www.mockdraftable.com//player/aj-brown...,2019,Wide Receiver,Ole Miss,"6' 0½""",226 lbs,"32⅞""","9¾""",,4.49s,"36½""","120""",,,19 reps
3,A.J. Cann,ajcann2015,https://www.mockdraftable.com//player/aj-cann?...,2015,Offensive Guard,South Carolina,"6' 3""",313 lbs,"32⅝""","10¼""",,,,,,,26 reps
4,A.J. Derby,ajderby2015,https://www.mockdraftable.com//player/aj-derby...,2015,Tight End,Arkansas,"6' 4""",255 lbs,"30½""","9½""",,,,,,,15 reps


In [18]:
nfl_combine_df.to_csv('nfl_combine_df_not_clean.csv', index=False)

In [19]:
nfl_combine_df.dtypes

player_name        object
player_id          object
player_url         object
draft_class         int64
position           object
college            object
height             object
weight             object
arm_length         object
hand_size          object
yard_10_split      object
yard_40_dash       object
vertical_jump      object
broad_jump         object
cone_drill         object
yard_20_shuttle    object
bench_press        object
dtype: object

In [20]:
FRACTIONS = {
    "½": ".5", "¼": ".25", "¾": ".75",
    "⅛": ".125", "⅜": ".375", "⅝": ".625", "⅞": ".875",
    "⅓": ".333", "⅔": ".667",
}

for col in nfl_combine_df.select_dtypes(include="object").columns:
    nfl_combine_df[col] = nfl_combine_df[col].astype(str).apply(replace_fractions)

In [21]:
nfl_combine_df["weight_lbs"] = nfl_combine_df["weight"].apply(extract_number)

In [22]:
nfl_combine_df["height_in"] = nfl_combine_df["height"].apply(parse_height)

In [23]:
for col in ["arm_length", "hand_size"]:
    nfl_combine_df[col + "_in"] = nfl_combine_df[col].apply(extract_number)

# 40-dash, 10-yd split, cone drill, 20-yd shuttle --> float seconds
for col in ["yard_40_dash", "yard_10_split", "cone_drill", "yard_20_shuttle"]:
    nfl_combine_df[col + "_sec"] = nfl_combine_df[col].apply(extract_number)

# vertical/broad jump--> decimal inches
for col in ["vertical_jump", "broad_jump"]:
    nfl_combine_df[col + "_in"] = nfl_combine_df[col].apply(extract_number)

# Bench press
nfl_combine_df["bench_press_reps"] = nfl_combine_df["bench_press"].apply(extract_number)

# Drop the original raw columns
raw_cols = [
    "weight", "height", "arm_length", "hand_size",
    "yard_40_dash", "yard_10_split", "cone_drill", "yard_20_shuttle",
    "vertical_jump", "broad_jump", "bench_press",
]
nfl_combine_df.drop(columns=raw_cols, inplace=True)


In [24]:
print(nfl_combine_df.dtypes)
print(nfl_combine_df.head())

player_name             object
player_id               object
player_url              object
draft_class              int64
position                object
college                 object
weight_lbs             float64
height_in              float64
arm_length_in          float64
hand_size_in           float64
yard_40_dash_sec       float64
yard_10_split_sec      float64
cone_drill_sec         float64
yard_20_shuttle_sec    float64
vertical_jump_in       float64
broad_jump_in          float64
bench_press_reps       float64
dtype: object
        player_name           player_id  \
0  A'Shawn Robinson  ashawnrobinson2016   
1      A.C. Leonard       acleonard2014   
2        A.J. Brown         ajbrown2019   
3         A.J. Cann          ajcann2015   
4        A.J. Derby         ajderby2015   

                                          player_url  draft_class  \
0  https://www.mockdraftable.com//player/ashawn-r...         2016   
1  https://www.mockdraftable.com//player/ac-leona...         2

In [25]:
display(nfl_combine_df.head())

,player_name,player_id,player_url,draft_class,position,college,weight_lbs,height_in,arm_length_in,hand_size_in,yard_40_dash_sec,yard_10_split_sec,cone_drill_sec,yard_20_shuttle_sec,vertical_jump_in,broad_jump_in,bench_press_reps
0,A'Shawn Robinson,ashawnrobinson2016,https://www.mockdraftable.com//player/ashawn-r...,2016,Defensive Tackle,Alabama,307.0,72.0,34.500,10.50,5.20,1.78,7.8,4.74,26.0,106.0,22.0
1,A.C. Leonard,acleonard2014,https://www.mockdraftable.com//player/ac-leona...,2014,Tight End,Tennessee State,252.0,72.0,33.000,9.25,4.50,NaN,NaN,NaN,34.0,128.0,20.0
2,A.J. Brown,ajbrown2019,https://www.mockdraftable.com//player/aj-brown...,2019,Wide Receiver,Ole Miss,226.0,72.0,32.875,9.75,4.49,NaN,NaN,NaN,36.5,120.0,19.0
3,A.J. Cann,ajcann2015,https://www.mockdraftable.com//player/aj-cann?...,2015,Offensive Guard,South Carolina,313.0,72.0,32.625,10.25,NaN,NaN,NaN,NaN,NaN,NaN,26.0
4,A.J. Derby,ajderby2015,https://www.mockdraftable.com//player/aj-derby...,2015,Tight End,Arkansas,255.0,72.0,30.500,9.50,NaN,NaN,NaN,NaN,NaN,NaN,15.0


In [26]:
nfl_combine_df.to_csv('nfl_combine_data.csv', index=False)